## 1. Imports

In [1]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt

from pytensor.graph import Apply, Op

from numba import njit, prange

print(f"Running on PyMC v{pm.__version__}")

Running on PyMC v5.26.1


## 2. Log likelihood of a timeseries using bridges + simulation function defs

In [2]:
def BDM_step(X0, k, mu, D, dt, timestepsperpoint):
    t = np.linspace(0, timestepsperpoint - 1, timestepsperpoint)
    X = np.zeros(timestepsperpoint)
    X[0] = X0

    dW = D * np.random.normal(size=timestepsperpoint)

    #Euler-Maruyama iteration
    for i in range(1,timestepsperpoint):
        X[i] = X[i-1] + k * (mu - X[i-1]) * dt + np.sqrt(X[i-1]) * dW[i-1]

    return X[-1]

def BDM_simulate_ts(X0, k, mu, D, dt, timestepsperpoint, N):
    datapoints = np.zeros(N)
    datapoints[0] = X0

    for idx in range(1,N):
        datapoints[idx] = BDM_step(datapoints[idx-1], k, mu, D, dt, timestepsperpoint)

    return datapoints

def SLM_step(X0, k, mu, D, dt, timestepsperpoint):
    t = np.linspace(0, timestepsperpoint - 1, timestepsperpoint)
    X = np.zeros(timestepsperpoint)
    X[0] = X0

    dW = D * np.random.normal(size=timestepsperpoint)

    #Euler-Maruyama iteration
    for i in range(1,timestepsperpoint):
        X[i] = X[i-1] + k * X[i-1] * (mu - X[i-1]) * dt + X[i-1] * dW[i-1]

    return X[-1]

def SLM_simulate_ts(X0, k, mu, D, dt, timestepsperpoint, N):
    datapoints = np.zeros(N)
    datapoints[0] = X0

    for idx in range(1,N):
        datapoints[idx] = SLM_step(datapoints[idx-1], k, mu, D, dt, timestepsperpoint)

    return datapoints

def simulate_timeseries(model='BDM', params=None):
    
    valid_models = {"BDM", "SLM"}
    X0, k, mu, D, dt, timestepsperpoint, N = params

    if params == None:
        raise ValueError("No parameters passed to simulate_timeseries()!")
    
    if model not in valid_models:
        raise ValueError(f"Invalid model: {model}. Must be one of: {', '.join(valid_models)}")
    elif model == 'BDM':
        sim_data = BDM_simulate_ts(X0, k, mu, D, dt, timestepsperpoint, N)
    elif model == 'SLM':
        sim_data = SLM_simulate_ts(X0, k, mu, D, dt, timestepsperpoint, N)
    return sim_data

@njit
def BDM_Lamperti(ts_data):
    arr = ts_data      
    # Check for negative values
    if np.any(arr < 0):
        raise ValueError("Input array contains negative values. Square root is not defined for negative numbers.")    
    result = 2 * np.sqrt(arr)
    return result   

@njit
def SLM_Lamperti(ts_data):
    arr = ts_data      
    # Check for negative values
    if np.any(arr < 0):
        raise ValueError("Input array contains negative values. Log is not defined for negative numbers.")    
    result = np.log(arr)
    return result

@njit
def bridge(NB, bridge_steps, x_i, x_f, Delta_t_B, D, random_numbers):        
    B = np.empty((NB, bridge_steps), dtype=np.float32) #to store simulated X values for bridges
    B[:, 0] = x_i
    B[:, -1] = x_f     
    for step in range(1, bridge_steps - 1): 
        #ATTENTION: Manal's script had (bridge_steps - step) * B[:, step - 1] + x_f with a PLUS!! Is this a grave typo in Javier's?
        mean_b = ((bridge_steps - step) * B[:, step - 1] + x_f ) / (bridge_steps - step + 1)        
        var_b = (D**2 * (bridge_steps - step) * Delta_t_B ) / (bridge_steps - step + 1)      
        B[:, step] = random_numbers[:, step - 1] * np.sqrt(var_b) + mean_b    
    return B

@njit
def I_hat(k, mu, B, Delta_t_B, model='BDM'):
    
    valid_models = {"BDM", "SLM"}

    # B = np.array(B)

    if B.ndim != 1:
        raise ValueError(f"Expected 1D array, but got {B.ndim}D array with shape {B.shape}")
    
    if model not in valid_models:
        raise ValueError(f"Invalid model: {model}. Must be one of: {', '.join(valid_models)}")
    elif model == 'BDM':
        Ihat = np.sum(((2 * k * mu - 0.5 * k * B**2 - 0.5) / B)**2) * Delta_t_B
    elif model == 'SLM':
        Ihat = np.sum((k * (mu - np.exp(B)) - 0.5)**2) * Delta_t_B

    return Ihat

@njit
def Gaussian_PDF(x, mean, std_dev):
    return (1 / (std_dev * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean) / std_dev)**2)

@njit
def log_RN_Derivative_BDM(B, k, mu, D, Delta_t_B, eff_dt):
    x_i = B[0]
    x_f = B[-1]

    Ihat = I_hat(k, mu, B, Delta_t_B, model='BDM')

    return (( 2 * k * mu * np.log(x_f/x_i) - 
              np.log(x_f/x_i) * 0.5 - 
              k * (x_f**2 - x_i**2) * 0.25 )/
            (D**2)) - (Ihat / ( 2 * (D**2) ) )

@njit
def log_RN_Derivative_SLM(B, k, mu, D, Delta_t_B, eff_dt):
    x_i = B[0]
    x_f = B[-1]

    Ihat = I_hat(k, mu, B, Delta_t_B, model='SLM')

    return (((k * mu - 0.5)*(x_f - x_i) - 
             k*(np.exp(x_f) - np.exp(x_i))
            ) / 
            (D**2)) - (Ihat / (2 * (D**2)))

@njit
def log_RN_Derivative(B, k, mu, D, Delta_t_B, eff_dt, model='BDM'):
    valid_models = {"BDM", "SLM"}

    if model not in valid_models:
        raise ValueError(f"Invalid model: {model}. Must be one of: {', '.join(valid_models)}")
    elif model == 'BDM':
        return log_RN_Derivative_BDM(B, k, mu, D, Delta_t_B, eff_dt)
    elif model == 'SLM':
        return log_RN_Derivative_SLM(B, k, mu, D, Delta_t_B, eff_dt)

@njit(parallel=True)
def logRND_each_bridge(bridges, k, mu, D, Delta_t_B, eff_dt, model='BDM'):

    # Pre-allocate the result array with the correct size
    n_bridges = len(bridges)
    logRND_array = np.empty(n_bridges)  # Pre-allocate
    
    # #Array of L_t for each bridge
    # logRND_array = np.array([])

    #For each bridge, calculate L and append to array
    for i in prange(n_bridges):
        bridge = bridges[i]
        logRND_array[i] = log_RN_Derivative(bridge, k, mu, D, Delta_t_B, eff_dt, model)

    return logRND_array

@njit
def est_log_propagator(bridges, k, mu, D, Delta_t_B, eff_dt, NB, x_i, x_f, model='BDM'):
    logRND_array = logRND_each_bridge(bridges, k, mu, D, Delta_t_B, eff_dt, model)
    lL_star = np.max(logRND_array)
    G = Gaussian_PDF(x_f, x_i, eff_dt)
    return lL_star + np.log(G) - np.log(NB) + np.log(np.sum(np.exp(logRND_array - lL_star)))

@njit(parallel=True)
def log_propagators_transformed_data(data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model='BDM'):

    # Pre-allocate result array
    n_pairs = len(data) - 1
    log_rho_array = np.empty(n_pairs)  # Pre-allocate with correct size

    for i in prange(n_pairs):
        x_i = data[i]
        x_f = data[i + 1]
        
        bridges = bridge(NB, bridge_steps, x_i, x_f, Delta_t_B, D, random_numbers)
        log_propagator = est_log_propagator(bridges, k, mu, D, Delta_t_B, eff_dt, NB, x_i, x_f, model)
        # log_rho_array = np.append(log_rho_array, log_propagator)
        log_rho_array[i] = log_propagator

    return log_rho_array

@njit
def log_propagators_target_process(log_rho_array, data, model='BDM'):
    valid_models = {"BDM", "SLM"}

    data = data[1:]

    if model not in valid_models:
        raise ValueError(f"Invalid model: {model}. Must be one of: {', '.join(valid_models)}")
    elif model == 'BDM':
        h_prime = 1 / np.sqrt(data)
        log_h_prime = np.log(h_prime)
        return log_h_prime + log_rho_array
    elif model == 'SLM':
        h_prime = 1 / data
        log_h_prime = np.log(h_prime)
        return log_h_prime + log_rho_array

@njit
def ts_logL(log_rho_array, p0):
    return np.sum(log_rho_array) + np.log(p0)

def full_logL_pipeline(sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code=0):
    
    models = ['BDM', 'SLM']
    model = models[model_code]
    
    lamperti_tf_data = SLM_Lamperti(sim_data)
    log_rho_tilde_array = log_propagators_transformed_data(
        lamperti_tf_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model
    )
    timeseries_logL = ts_logL(log_propagators_target_process(log_rho_tilde_array, sim_data, 'SLM'), 1.0)
    return timeseries_logL

### 2.1. Compiling Numba

Numba compiles the decorated functions the first time it is called. We do this here, before any significant code is run.

In [4]:
# Parameters
MODEL = 'SLM'
model_code = 1
X0 = 4.8
k = 3
mu = 5
D = 0.005
dt = 3e-3 #timestep of the simulation
T = 10 #timeseries timespan, starts at 0 and ends at T
eff_dt = 1 #timestep of the datapoints
N = round(T+1 / eff_dt) #number of datapoints
timestepsperpoint = round(eff_dt / dt)
NB = 8000  # Number of bridges per transition
timestep_ratio = 1e3 # dt / Dt
bridge_steps = int(eff_dt * timestep_ratio) # Number of time steps inside bridge
Delta_t_B = eff_dt / bridge_steps 
datapoint_times = np.linspace(0, T, N)

In [5]:
# Outside Numba (in Python)
rng = np.random.default_rng(seed=42)
random_numbers = rng.random((NB, bridge_steps - 2))

In [6]:
sim_data = simulate_timeseries(model=MODEL, params=(X0, k, mu, D, dt, timestepsperpoint, N))

In [8]:
loglogl = full_logL_pipeline(sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code)
print(loglogl)

-88858.16105861896


## 3. PyTensor Op

In [184]:
# # define a pytensor Op for our likelihood function

# class timeseries_LogLike_op(Op):
#     def make_node(self, sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code) -> Apply:
#         # Convert inputs to tensor variables
#         sim_data = pt.as_tensor(sim_data)
#         NB = pt.as_tensor(NB)
#         bridge_steps = pt.as_tensor(bridge_steps)
#         Delta_t_B = pt.as_tensor(Delta_t_B)
#         D = pt.as_tensor(D)
#         k = pt.as_tensor(k)
#         mu = pt.as_tensor(mu)
#         random_numbers = pt.as_tensor(random_numbers)
#         model_code = pt.as_tensor(model_code)
        

#         inputs = [sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code]
#         # Define output type, in our case a vector of likelihoods
#         # with the same dimensions and same data type as data
#         # If data must always be a vector, we could have hard-coded
#         # outputs = [pt.vector()]
#         outputs = [Delta_t_B.type()]

#         # Apply is an object that combines inputs, outputs and an Op (self)
#         return Apply(self, inputs, outputs)

#     def perform(self, node: Apply, inputs: list[np.ndarray], outputs: list[list[None]]) -> None:
#         # This is the method that compute numerical output
#         # given numerical inputs. Everything here is numpy arrays
#         sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code = inputs  # this will contain my variables

#         sim_data = np.asarray(sim_data, dtype=np.float64)
#         NB = int(np.asarray(NB, dtype=np.int64))  # Extract scalar
#         bridge_steps = int(np.asarray(bridge_steps, dtype=np.int64))  # Extract scalar
#         Delta_t_B = float(np.asarray(Delta_t_B, dtype=np.float64))
#         D = float(np.asarray(D, dtype=np.float64))
#         k = float(np.asarray(k, dtype=np.float64))
#         mu = float(np.asarray(mu, dtype=np.float64))
#         random_numbers = np.asarray(random_numbers, dtype=np.float64)
#         model_code = int(np.asarray(model_code, dtype=np.int64))


#         # call our numpy log-likelihood function
#         loglike_eval = full_logL_pipeline(sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code)

#         # Save the result in the outputs list provided by PyTensor
#         # There is one list per output, each containing another list
#         # pre-populated with a `None` where the result should be saved.
#         outputs[0][0] = np.asarray(loglike_eval)

In [9]:
# Pre-convert fixed constants to PyTensor constants
random_numbers_tensor = pt.as_tensor(random_numbers, dtype="float64")
NB_const = pt.as_tensor(NB, dtype="int64")
bridge_steps_const = pt.as_tensor(bridge_steps, dtype="int64")
Delta_t_B_const = pt.as_tensor(Delta_t_B, dtype="float64")
D_const = pt.as_tensor(D, dtype="float64")
model_code_const = pt.as_tensor(model_code, dtype="int64")

# Define the Op with fixed constants baked in
class timeseries_LogLike_op(Op):
    def __init__(self, random_numbers_tensor, NB, bridge_steps, Delta_t_B, D, model_code):
        self.random_numbers_tensor = random_numbers_tensor
        self.NB = NB
        self.bridge_steps = bridge_steps
        self.Delta_t_B = Delta_t_B
        self.D = D
        self.model_code = model_code

    def make_node(self, sim_data, k, mu) -> Apply:
        sim_data = pt.as_tensor(sim_data)
        k = pt.as_tensor(k)
        mu = pt.as_tensor(mu)
        outputs = [pt.scalar()]  # Scalar log-likelihood
        return Apply(self, [sim_data, k, mu], outputs)

    def perform(self, node: Apply, inputs: list[np.ndarray], outputs: list[list[None]]) -> None:
        sim_data, k, mu = inputs
        sim_data = np.asarray(sim_data, dtype=np.float64)
        k = float(np.asarray(k, dtype=np.float64))
        mu = float(np.asarray(mu, dtype=np.float64))

        # Use the pre-baked constants
        NB = self.NB
        bridge_steps = self.bridge_steps
        Delta_t_B = self.Delta_t_B
        D = self.D
        model_code = self.model_code
        random_numbers = self.random_numbers_tensor.eval()  # This is safe because it's a constant

        # Call your pipeline
        loglike_eval = full_logL_pipeline(
            sim_data, NB, bridge_steps, Delta_t_B, D, k, mu, random_numbers, model_code
        )

        outputs[0][0] = np.asarray(loglike_eval, dtype=np.float64)

## 4. Pymc Model

In [10]:
with pm.Model() as no_grad_model:
    k = pm.Uniform("k", lower=1., upper=8., initval=3.)
    mu = pm.Uniform("mu", lower=2., upper=10., initval=5.)

    # Use the Op with fixed constants baked in
    loglike_op = timeseries_LogLike_op(
        random_numbers_tensor=random_numbers_tensor,
        NB=NB,
        bridge_steps=bridge_steps,
        Delta_t_B=Delta_t_B,
        D=D,
        model_code=model_code
    )

    likelihood = pm.CustomDist(
        "likelihood",
        k,  # dist_param
        mu,  # dist_param
        observed=sim_data,
        logp=loglike_op,
        # No need to pass NB, bridge_steps, etc. — they're in the Op
    )

In [11]:
ip = no_grad_model.initial_point()
ip

{'k_interval__': array(-0.91629073), 'mu_interval__': array(-0.51082562)}

In [12]:
no_grad_model.compile_logp(vars=[likelihood], sum=False)(ip)

/tmp/ipykernel_52272/1823891312.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  k = float(np.asarray(k, dtype=np.float64))
/tmp/ipykernel_52272/1823891312.py:30: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mu = float(np.asarray(mu, dtype=np.float64))


[array(-88858.16105862)]

In [14]:
with no_grad_model:
    idata_no_grad = pm.sample(
        draws=3000,
        tune=1000,
        cores=4,
        mp_ctx='spawn',  # ← This is the fix!
        random_seed=42
    )

# # plot the traces
# az.plot_trace(idata_no_grad, lines=[("m", {}, mtrue), ("c", {}, ctrue)]);

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Slice: [k]
>Slice: [mu]


Output()

/tmp/ipykernel_52272/1823891312.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_52272/1823891312.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_52272/1823891312.py:30: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_52272/1823891312.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing t

ValueError: Not enough samples to build a trace.

## 5. True Parameters

## 6. Timeseries simulation

## 7. Parameter Inference